# ERM baseline: fresh classifier on the same Waterbirds-ERM embeddings

This notebook is the **no-Rater control** for your frozen-Rater final-classifier experiment.

It keeps the comparison controlled:
- same Waterbirds-ERM 2048-D embeddings;
- same `val_subset1` training split;
- same `val_subset2` WGA model selection;
- same test split;
- same fresh random `Linear(2048, 2)` classifier;
- same SGD hyperparameters and 50 epochs.

The only difference is that there is **no Rater and no reweighting**:

\[
L_{ERM}=rac{1}{B}\sum_i CE_i.
\]

Use the **restored test accuracy/WGA** after selecting the epoch on `val_subset2` for the fair comparison.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import json, csv, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DRIVE_ROOT = Path('/content/drive/MyDrive/evidential_alignment_colab')
EMBEDDING_CACHE_ROOT = Path(
    '/content/drive/MyDrive/evidential_alignment_colab/'
    'embeddings_waterbirds_erm/waterbirds/'
    'resnet50_waterbirds_erm_1055b1e8de29/resolution_224'
)
EXPECTED_ERM_FINGERPRINT = '1055b1e8de29'

SEED = 0
SPLIT_VAL = 0.6
BATCH_SIZE = 128
NUM_WORKERS = 2
FINAL_EPOCHS = 50
FINAL_LR = 1e-2
FINAL_MOMENTUM = 0.9
FINAL_WEIGHT_DECAY = 1e-4
SELECTION_METRIC = 'wga'  # 'wga' or 'accuracy'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

RUNS_ROOT = DRIVE_ROOT / 'erm_fresh_classifier_baselines'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT = RUNS_ROOT / (
    f'fresh_linear_erm_splitval_{SPLIT_VAL}_lr_{FINAL_LR}_'
    f'{datetime.now().strftime("%Y%m%d_%H%M%S")}'
)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
PLOTS_ROOT = RUN_ROOT / 'plots'
PLOTS_ROOT.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_ROOT / 'log.txt'
HISTORY_PATH = RUN_ROOT / 'history.csv'

assert EMBEDDING_CACHE_ROOT.exists()
print('Device:', DEVICE)
print('Embedding cache:', EMBEDDING_CACHE_ROOT)
print('Run directory:', RUN_ROOT)

Mounted at /content/drive
Device: cpu
Embedding cache: /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224
Run directory: /content/drive/MyDrive/evidential_alignment_colab/erm_fresh_classifier_baselines/fresh_linear_erm_splitval_0.6_lr_0.01_20261007_114134


In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def log(message=''):
    message = str(message)
    print(message, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(message + '\n')

config = {
    'experiment': 'fresh_linear_erm_no_rater',
    'embedding_cache_root': str(EMBEDDING_CACHE_ROOT),
    'expected_erm_fingerprint': EXPECTED_ERM_FINGERPRINT,
    'seed': SEED,
    'split_val': SPLIT_VAL,
    'batch_size': BATCH_SIZE,
    'epochs': FINAL_EPOCHS,
    'lr': FINAL_LR,
    'momentum': FINAL_MOMENTUM,
    'weight_decay': FINAL_WEIGHT_DECAY,
    'selection_metric': SELECTION_METRIC,
}
with open(RUN_ROOT / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

log('=' * 90)
log('ERM BASELINE: FRESH LINEAR CLASSIFIER / NO RATER')
log('=' * 90)
log(json.dumps(config, indent=2))

ERM BASELINE: FRESH LINEAR CLASSIFIER / NO RATER
{
  "experiment": "fresh_linear_erm_no_rater",
  "embedding_cache_root": "/content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224",
  "expected_erm_fingerprint": "1055b1e8de29",
  "seed": 0,
  "split_val": 0.6,
  "batch_size": 128,
  "epochs": 50,
  "lr": 0.01,
  "momentum": 0.9,
  "weight_decay": 0.0001,
  "selection_metric": "wga"
}


In [3]:
class IndexedEmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels, groups, attrs, split_name):
        self.embeddings = embeddings.float()
        self.labels = labels.long()
        self.groups = groups.long()
        self.attrs = attrs.long()
        self.split_name = split_name

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return idx, self.embeddings[idx], self.labels[idx], self.groups[idx], self.attrs[idx]

def tok(x):
    return str(x).replace('.', 'p')

def split_path(split_name):
    if split_name in {'train_no_aug', 'val', 'test'}:
        return EMBEDDING_CACHE_ROOT / f'{split_name}.pt'
    if split_name in {'val_subset1', 'val_subset2'}:
        return EMBEDDING_CACHE_ROOT / (
            f'{split_name}_splittrain_1p0_splitval_{tok(SPLIT_VAL)}_seed_{SEED}.pt'
        )
    raise ValueError(split_name)

def ensure_val_subsets():
    p1 = split_path('val_subset1')
    p2 = split_path('val_subset2')
    if p1.exists() and p2.exists():
        log('[Embeddings] requested val subsets already exist.')
        return

    full_val_path = split_path('val')
    payload = torch.load(full_val_path, map_location='cpu', weights_only=False)
    n = int(payload['labels'].shape[0])
    rng = np.random.RandomState(SEED)
    indices = np.arange(n)
    rng.shuffle(indices)
    n1 = int(n * SPLIT_VAL)
    split_indices = {
        'val_subset1': torch.as_tensor(indices[:n1], dtype=torch.long),
        'val_subset2': torch.as_tensor(indices[n1:], dtype=torch.long),
    }

    for split_name, idx in split_indices.items():
        target = split_path(split_name)
        subset = {
            'embeddings': payload['embeddings'][idx].clone(),
            'labels': payload['labels'][idx].clone(),
            'groups': payload['groups'][idx].clone(),
            'attrs': payload['attrs'][idx].clone(),
            'cache_metadata': dict(payload.get('cache_metadata', {})),
        }
        subset['cache_metadata'].update({
            'split': split_name,
            'split_train': 1.0,
            'split_val': float(SPLIT_VAL),
            'seed': int(SEED),
            'derived_from_full_val_cache': True,
        })
        torch.save(subset, target)
        log(f'[Embeddings] created {split_name}: n={len(idx)}')

ensure_val_subsets()

def load_split(split_name):
    path = split_path(split_name)
    payload = torch.load(path, map_location='cpu', weights_only=False)
    z = payload['embeddings']
    assert z.ndim == 2 and z.shape[1] == 2048

    meta = payload.get('cache_metadata', {})
    rep = meta.get('representation_source')
    if rep and rep != 'waterbirds_erm_checkpoint':
        raise RuntimeError(f'{split_name} is not Waterbirds-ERM embeddings.')

    fp = meta.get('erm_checkpoint_fingerprint') or meta.get('erm_fingerprint')
    if fp and fp != EXPECTED_ERM_FINGERPRINT:
        raise RuntimeError(
            f'{split_name} fingerprint mismatch: {fp} != {EXPECTED_ERM_FINGERPRINT}'
        )

    ds = IndexedEmbeddingDataset(
        payload['embeddings'], payload['labels'], payload['groups'], payload['attrs'], split_name
    )
    log(f'[Embeddings] {split_name}: n={len(ds)}')
    return ds

training_split = 'val_subset1'
selection_split = 'val_subset2'
test_split = 'test'

training_dataset = load_split(training_split)
selection_dataset = load_split(selection_split)
test_dataset = load_split(test_split)

for name, ds in [('training', training_dataset), ('selection', selection_dataset), ('test', test_dataset)]:
    log('')
    log(f'{name}: n={len(ds)}')
    groups = ds.groups.numpy()
    for gid in sorted(np.unique(groups)):
        log(f'  g{gid}: {int((groups == gid).sum())}')

[Embeddings] requested val subsets already exist.
[Embeddings] val_subset1: n=719
[Embeddings] val_subset2: n=480
[Embeddings] test: n=5794

training: n=719
  g0: 272
  g1: 281
  g2: 78
  g3: 88

selection: n=480
  g0: 195
  g1: 185
  g2: 55
  g3: 45

test: n=5794
  g0: 2255
  g1: 2255
  g2: 642
  g3: 642


In [4]:
class LinearClassifier(nn.Module):
    def __init__(self, input_dim=2048, num_classes=2):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, z):
        return self.linear(z)

@torch.no_grad()
def evaluate(model, dataset):
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_n = 0
    group_correct = {}
    group_total = {}

    for _, z, y, g, _ in loader:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        g = g.to(DEVICE, non_blocking=True)
        logits = model(z)
        pred = logits.argmax(dim=1)
        total_loss += float(F.cross_entropy(logits, y, reduction='sum').item())
        total_correct += int(pred.eq(y).sum().item())
        total_n += int(y.numel())

        for gid_t in torch.unique(g):
            gid = int(gid_t.item())
            mask = g.eq(gid_t)
            group_correct[gid] = group_correct.get(gid, 0) + int(
                pred[mask].eq(y[mask]).sum().item()
            )
            group_total[gid] = group_total.get(gid, 0) + int(mask.sum().item())

    group_accuracy = {
        gid: group_correct[gid] / group_total[gid]
        for gid in sorted(group_total)
    }
    return {
        'loss': total_loss / max(total_n, 1),
        'accuracy': total_correct / max(total_n, 1),
        'group_accuracy': group_accuracy,
        'wga': min(group_accuracy.values()),
    }

def selection_value(metrics):
    if SELECTION_METRIC == 'wga':
        return metrics['wga']
    if SELECTION_METRIC == 'accuracy':
        return metrics['accuracy']
    raise ValueError(SELECTION_METRIC)

In [5]:
set_seed(SEED)

classifier = LinearClassifier(2048, 2).to(DEVICE)
optimizer = torch.optim.SGD(
    classifier.parameters(),
    lr=FINAL_LR,
    momentum=FINAL_MOMENTUM,
    weight_decay=FINAL_WEIGHT_DECAY,
)

train_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

best_value = -float('inf')
best_epoch = None
best_state = None
best_selection_metrics = None
oracle_test_wga = -float('inf')
oracle_test_epoch = None
history = []

for epoch in range(1, FINAL_EPOCHS + 1):
    classifier.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_n = 0

    for _, z, y, _, _ in train_loader:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = classifier(z)

        # ORDINARY ERM: every sample has equal weight.
        loss = F.cross_entropy(logits, y, reduction='mean')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_n = int(y.numel())
        train_loss_sum += float(loss.item()) * batch_n
        train_correct += int(logits.argmax(dim=1).eq(y).sum().item())
        train_n += batch_n

    train_loss = train_loss_sum / max(train_n, 1)
    train_acc = train_correct / max(train_n, 1)

    val_metrics = evaluate(classifier, selection_dataset)
    test_metrics = evaluate(classifier, test_dataset)
    current_value = selection_value(val_metrics)

    if current_value > best_value:
        best_value = float(current_value)
        best_epoch = int(epoch)
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in classifier.state_dict().items()
        }
        best_selection_metrics = dict(val_metrics)

    if test_metrics['wga'] > oracle_test_wga:
        oracle_test_wga = float(test_metrics['wga'])
        oracle_test_epoch = int(epoch)

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_accuracy': train_acc,
        'selection_accuracy': val_metrics['accuracy'],
        'selection_wga': val_metrics['wga'],
        'test_accuracy': test_metrics['accuracy'],
        'test_wga': test_metrics['wga'],
    }
    history.append(row)

    log(
        f'[epoch {epoch:03d}/{FINAL_EPOCHS}] '
        f'train_loss={train_loss:.6f} | '
        f'train_acc={100*train_acc:.2f}% | '
        f'val_acc={100*val_metrics["accuracy"]:.2f}% | '
        f'val_WGA={100*val_metrics["wga"]:.2f}% | '
        f'test_acc={100*test_metrics["accuracy"]:.2f}% | '
        f'test_WGA={100*test_metrics["wga"]:.2f}%'
    )

assert best_state is not None
classifier.load_state_dict(best_state)
classifier.to(DEVICE)
classifier.eval()
restored_test_metrics = evaluate(classifier, test_dataset)

torch.save(
    {
        'model_sd': classifier.state_dict(),
        'experiment': 'fresh_linear_erm_no_rater',
        'selected_epoch': best_epoch,
        'selection_metric': SELECTION_METRIC,
        'selection_metrics': best_selection_metrics,
        'test_metrics': restored_test_metrics,
        'embedding_cache_root': str(EMBEDDING_CACHE_ROOT),
        'training_split': training_split,
        'selection_split': selection_split,
        'test_split': test_split,
    },
    RUN_ROOT / 'best_erm_classifier.pt',
)

with open(HISTORY_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)

log('')
log('=' * 90)
log(f'SELECTED EPOCH = {best_epoch}')
log(f'selection accuracy = {100*best_selection_metrics["accuracy"]:.2f}%')
log(f'selection WGA = {100*best_selection_metrics["wga"]:.2f}%')
log(f'restored test accuracy = {100*restored_test_metrics["accuracy"]:.2f}%')
log(f'restored test WGA = {100*restored_test_metrics["wga"]:.2f}%')
log(f'test-oracle WGA = {100*oracle_test_wga:.2f}% at epoch {oracle_test_epoch}')
log('=' * 90)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[epoch 001/50] train_loss=0.435842 | train_acc=77.89% | val_acc=91.88% | val_WGA=50.91% | test_acc=91.72% | test_WGA=53.58%
[epoch 002/50] train_loss=0.183134 | train_acc=92.35% | val_acc=93.96% | val_WGA=81.82% | test_acc=94.34% | test_WGA=85.67%
[epoch 003/50] train_loss=0.162794 | train_acc=92.91% | val_acc=95.42% | val_WGA=78.18% | test_acc=95.24% | test_WGA=84.74%
[epoch 004/50] train_loss=0.147187 | train_acc=94.44% | val_acc=95.00% | val_WGA=70.91% | test_acc=95.43% | test_WGA=81.46%
[epoch 005/50] train_loss=0.141857 | train_acc=94.58% | val_acc=95.62% | val_WGA=74.55% | test_acc=95.69% | test_WGA=83.80%
[epoch 006/50] train_loss=0.126297 | train_acc=94.85% | val_acc=95.83% | val_WGA=83.64% | test_acc=95.50% | test_WGA=87.07%
[epoch 007/50] train_loss=0.118405 | train_acc=95.41% | val_acc=96.04% | val_WGA=83.64% | test_acc=95.60% | test_WGA=87.07%
[epoch 008/50] train_loss=0.104571 | train_acc=95.55% | val_acc=96.25% | val_WGA=80.00% | test_acc=95.93% | test_WGA=86.14%
[epoch 0

In [6]:
epochs = [row['epoch'] for row in history]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, [row['selection_accuracy'] for row in history], label='selection accuracy')
ax.plot(epochs, [row['selection_wga'] for row in history], label='selection WGA')
ax.plot(epochs, [row['test_accuracy'] for row in history], label='test accuracy', alpha=0.75)
ax.plot(epochs, [row['test_wga'] for row in history], label='test WGA', alpha=0.75)
ax.axvline(best_epoch, linestyle='--', label=f'selected epoch {best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Fresh linear ERM baseline on Waterbirds-ERM embeddings')
ax.grid(True, linestyle=':', alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'accuracy_wga.png', dpi=160, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, [row['train_loss'] for row in history])
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean cross-entropy')
ax.set_title('ERM training loss')
ax.grid(True, linestyle=':', alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'train_loss.png', dpi=160, bbox_inches='tight')
plt.close(fig)

summary = {
    'experiment': 'fresh_linear_erm_no_rater',
    'training_split': training_split,
    'selection_split': selection_split,
    'test_split': test_split,
    'selected_epoch': best_epoch,
    'selection_metric': SELECTION_METRIC,
    'selection_metrics': best_selection_metrics,
    'restored_test_metrics': restored_test_metrics,
    'test_oracle_wga': oracle_test_wga,
    'test_oracle_epoch': oracle_test_epoch,
}
with open(RUN_ROOT / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

with open(RUN_ROOT / 'summary.txt', 'w', encoding='utf-8') as f:
    f.write('FRESH LINEAR ERM BASELINE\n')
    f.write('=' * 80 + '\n')
    f.write('Rater used = NO\n')
    f.write(f'Training split = {training_split}\n')
    f.write(f'Selection split = {selection_split}\n')
    f.write(f'Test split = {test_split}\n')
    f.write(f'Selected epoch = {best_epoch}\n')
    f.write(f'Selection accuracy = {best_selection_metrics["accuracy"]:.8f}\n')
    f.write(f'Selection WGA = {best_selection_metrics["wga"]:.8f}\n')
    f.write(f'Restored test accuracy = {restored_test_metrics["accuracy"]:.8f}\n')
    f.write(f'Restored test WGA = {restored_test_metrics["wga"]:.8f}\n')
    f.write(f'Test-oracle WGA = {oracle_test_wga:.8f} at epoch {oracle_test_epoch}\n')

print('Done.')
print('Run directory:', RUN_ROOT)
print('Best classifier:', RUN_ROOT / 'best_erm_classifier.pt')
print('Summary:', RUN_ROOT / 'summary.txt')

Done.
Run directory: /content/drive/MyDrive/evidential_alignment_colab/erm_fresh_classifier_baselines/fresh_linear_erm_splitval_0.6_lr_0.01_20261007_114134
Best classifier: /content/drive/MyDrive/evidential_alignment_colab/erm_fresh_classifier_baselines/fresh_linear_erm_splitval_0.6_lr_0.01_20261007_114134/best_erm_classifier.pt
Summary: /content/drive/MyDrive/evidential_alignment_colab/erm_fresh_classifier_baselines/fresh_linear_erm_splitval_0.6_lr_0.01_20261007_114134/summary.txt


## Fair comparison

Compare the **restored test metrics** from this notebook with the restored test
metrics from the frozen-Rater notebook.

ERM baseline:
```text
fresh Linear(2048,2)
+ same ERM embeddings
+ val_subset1 training
+ ordinary CE
+ val_subset2 WGA selection
```

Frozen-Rater experiment:
```text
fresh Linear(2048,2)
+ same ERM embeddings
+ val_subset1 training
+ Rater-weighted CE
+ val_subset2 WGA selection
```

The controlled difference is the Rater weighting. `test_oracle_wga` is kept only
as a diagnostic and should not be the primary reported result.